# 🛠️ netctl — the hands that touch the router

In `08_controller_the_bouncer.ipynb` the controller said *yes* and called
`provisioner.apply_bandwidth(...)` through the `NetworkProvisioner` port you met in
`03_protocols_and_ports.ipynb` — and that was the last you saw of the call. This chapter
opens the other side of that port: **netctl**, the small library that turns "Ada bought
50 Mbps" into actual configuration on an actual router.

**What you'll be able to do after this notebook**

- Explain what "provisioning" means from zero, and why routers grew a programmable API (gNMI) to replace humans typing at a screen
- Read a YANG path like `/qos/policer-templates/policer-template[name=a2a-7]` and say why every such string lives in one file
- Drive `MockProvisioner` live and prove rule 8 (idempotent teardown) with your own calls
- Use the shared contract-test suite (rule 7) to catch a lying mock that `isinstance` waves through
- Walk `GnmiProvisioner`'s real gNMI code and state *honestly* what this lab does and doesn't enforce (ADR-006)

**You need:** notebooks 00–08. No router, no chain, no LLM — `MockProvisioner` carries
the whole chapter. One optional section detects the containerlab lab and politely skips
(printing the recipe) when it's not running.

**Estimated time:** 60–90 minutes.

> The compact live-lab tour of the same code is
> [`e2e/notebooks/netctl_explore.ipynb`](../netctl_explore.ipynb) — replay it *after*
> this chapter, with the lab up, and everything in it will feel obvious.

## 1. The last inch: from ticket to physics

Recap the story's state: Ada paid 10 TOK, ticket #7 exists on chain, the bouncer checked
every fact and said yes (08). And yet — nothing in the world has changed. No packet moves
faster or slower. The promise is still only paperwork.

Three words from zero, because everything below stands on them:

- A **router** is a computer whose one job is forwarding packets between networks. Bell's
  router in this lab is `srl1`, a containerized Nokia **SR Linux** — a real network
  operating system, the same software that runs on physical hardware.
- An **interface** is one of the router's network sockets — `ethernet-1/1`, `ethernet-1/2`.
  (Networking calls these "ports" too; that's the *third* meaning of the word after 03's
  architecture-port and 08's TCP-port. We'll say *interface*.)
- The router's **config** is its settings database: which interfaces are up, what
  addresses they carry, what rate limits apply. Change the config, change the physics.

**Provisioning** is exactly that last step: writing config so a paid-for promise becomes
enforced reality. netctl is this repo's hands — the only package that touches routers.

The lab the hands touch is three containers wired like a tiny internet — the file below
is **containerlab** topology-as-code (a tool that boots network labs from a YAML file).
Read the ASCII picture in its header: Ada's traffic (hostA) *must* cross `srl1` to reach
hostB, which is precisely why a rate limit installed on `srl1` can shape her whole flow.

In [ ]:
from pathlib import Path

# repo root, found by walking up to the .git directory (course cwd is 3 levels deep)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())

topo = (ROOT / "netlab/topology.clab.yml").read_text()
print(topo[: topo.index("name: a2a")].rstrip())          # the header comment = the map

assert "hostA" in topo and "srl1" in topo and "hostB" in topo
print("\n✓ three nodes: hostA ── srl1 ── hostB   ← every packet crosses the router")

## 2. The old world: a human at the router's keyboard

For decades, "provisioning" meant a human opening a **CLI** (command-line interface) on
the router and typing configuration commands, line by line. This repo did that too — on
purpose. Milestone M2.2 derived the 50 Mbps rate limit *by hand* before any code
automated it, and the exact keystrokes are preserved in `docs/07-netlab.md` §6.1.

The cell below pulls that recipe out of the doc. Read it as two moves:

1. create a **policer template** — *policer* is router-speak for a rate limiter — named
   `police-50m`, with peak and committed rates of 50,000 kbps (= 50 Mbps);
2. **attach** it to the input of `ethernet-1/1.0` — the edge where Ada's traffic enters
   Bell's network. You police at the border, not in the core.

Now imagine an AI agent doing this: generating CLI text, pasting it into a terminal
emulator, and parsing whatever prose the router prints back. Fragile, unparseable,
vendor-specific. Machines needed an actual API.

In [ ]:
doc = (ROOT / "docs/07-netlab.md").read_text()
start = doc.index("enter candidate\nset / qos policer-templates")
end = doc.index("commit now", start) + len("commit now")
print(doc[start:end])

assert "police-50m" in doc[start:end]
print("\n✓ the M2.2 hand recipe — keep it in view; section 13 shows it reborn as Python")

**✏️ Your turn 2.1 — count the config writes (predict, then check)**

The recipe you just read is already sliced into `doc[start:end]`. Predict — *before*
running — how many of its lines start with `set /` (the config-writing verb), then
un-comment the assert. Success: your prediction is right, and you can point at which
of those lines is the *attach* move from the section's move 2.

In [ ]:
recipe = doc[start:end]                       # §2's hand recipe, already sliced

predicted_set_lines = 0                       # TODO: your prediction BEFORE running

set_lines = [ln for ln in recipe.splitlines() if ln.startswith("set /")]
print(f"{len(set_lines)} `set /` lines; the last one:")
print("  ", set_lines[-1])

# un-comment once your prediction matches:
# assert len(set_lines) == 4 and "input policer-templates" in set_lines[-1]

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
predicted_set_lines = 4
assert len(set_lines) == 4 and "input policer-templates" in set_lines[-1]
```

One line builds the template (with two `\`-continuations that *don't* start with
`set /`), two wire the subinterface reference, and the last — `input
policer-templates` — is the attach: policing lands at the border where Ada's traffic
enters.

</details>

## 3. gNMI — Get, Set, Subscribe

The industry's answer is **gNMI** (gRPC Network Management Interface): a standard API
that network devices expose so *programs* can manage them. It rides on **gRPC** — a way
for a program to call functions on another machine, like the HTTP APIs you met in 08 but
binary and streaming-friendly. gNMI has essentially three verbs:

| verb | meaning | netctl uses it for |
|---|---|---|
| **Get** | read config or live state, once | finding its own work before teardown; health checks |
| **Set** | change config (transactionally) | installing/removing policers and telemetry exports |
| **Subscribe** | "push me updates on a schedule" | the raw telemetry stream (seen by hand in docs/07 §7) |

That's the whole vocabulary. Everything `NetworkProvisioner` promises maps onto these
verbs — run the cell to see the mapping against the real port from 03.

In [ ]:
import inspect

from a2a_interfaces import NetworkProvisioner

mapping = {
    "apply_bandwidth": "Set            (write policer template + attachment)",
    "apply_telemetry": "Set            (write a telemetry export destination)",
    "teardown":        "Get + Set-delete (find own config on the device, remove it)",
    "health":          "Capabilities    (the gNMI hello — 'are you there?')",
}
for name, verb in mapping.items():
    assert hasattr(NetworkProvisioner, name)             # the port really promises it
    print(f"{name:16} → {verb}")

print("\n✓ four port methods, three gNMI verbs — the whole surface of 'the hands'")

## 4. YANG paths — a filesystem for config (`paths.py`)

gNMI's verbs need addresses: *which* piece of config to Get or Set. Those addresses come
from **YANG** — a schema language in which the router's vendor defines every config knob
as a node in one giant tree. A **YANG path** is a slash-separated address into that tree,
exactly like a filesystem path:

```
/qos/policer-templates/policer-template[name=a2a-7]
└┬─┘ └───────┬───────┘ └──────┬───────┘└────┬─────┘
 dir      subdir        a LIST of them   which list entry
```

The `[name=a2a-7]` part is the **key predicate** — YANG lists hold many entries, and the
bracket picks one by its key, the way `d["a2a-7"]` picks a dict entry. That syntax lives
*inside* the string; Python just sees characters.

netctl keeps every path it ever touches in one tiny module, `netctl/src/netctl/paths.py`.
Here is the whole file — read the docstring first, it states the rule this file *is*.

In [ ]:
from netctl import paths

print(inspect.getsource(paths))

Three things to take from that file:

- **Constants for tree roots** (`QOS_POLICER_TEMPLATES`, …) and **functions for
  addressed entries** — a function like `policer_template(name)` is just an f-string
  (01) wrapping the key predicate, so callers can't typo the bracket syntax.
- The section comments tie each path to the *product* it serves: the policer paths are
  the bandwidth product, `grpc-tunnel` is the telemetry product (§19).
- Nothing else: no I/O, no classes. Pure string-building — which is why this cell runs
  on any machine, router or not.

Poke the builders with the canonical names. `a2a-7` is what session naming will look
like for ticket #7's session; `ethernet-1/1.0` is the subinterface from the hand recipe.

In [ ]:
surface = sorted(n for n in vars(paths) if not n.startswith("_") and n != "annotations")
print("the whole module surface:", surface, "\n")

p1 = paths.policer_template("a2a-7")
p2 = paths.qos_interface("ethernet-1/1.0")
p3 = paths.interface_oper_state("ethernet-1/1")
p4 = paths.telemetry_destination("a2a-8")
for p in (p1, p2, p3, p4):
    print(" ", p)

assert p1 == "/qos/policer-templates/policer-template[name=a2a-7]"
assert p2 == "/qos/interfaces/interface[interface-id=ethernet-1/1.0]"
assert p3 == "/interface[name=ethernet-1/1]/oper-state"
assert p4 == "/system/grpc-tunnel/destination[name=a2a-8]"
print("\n✓ four addresses into srl1's config tree — note the two different key names")

**✏️ Your turn 4.1 — address ticket #8's config yourself**

Story chapter 7 sells a *second* session (ticket #8), and suppose its policer must sit
on `ethernet-1/2.0` instead. Fix the two deliberately-wrong builder calls so they
address session `a2a-8` and that interface. Success: both un-commented asserts pass
without you typing a single `[` yourself.

In [ ]:
# TODO: fix the argument of each builder (session a2a-8, interface ethernet-1/2.0)
t8 = paths.policer_template("a2a-7")          # ← wrong session on purpose
a8 = paths.qos_interface("ethernet-1/1.0")    # ← wrong interface on purpose
print("template   →", t8)
print("attachment →", a8)

# un-comment once both are fixed:
# assert t8 == "/qos/policer-templates/policer-template[name=a2a-8]"
# assert a8 == "/qos/interfaces/interface[interface-id=ethernet-1/2.0]"

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
t8 = paths.policer_template("a2a-8")
a8 = paths.qos_interface("ethernet-1/2.0")
assert t8 == "/qos/policer-templates/policer-template[name=a2a-8]"
assert a8 == "/qos/interfaces/interface[interface-id=ethernet-1/2.0]"
```

The builders own the bracket syntax (and the two *different* key names, `name` vs
`interface-id`) — callers only ever supply the value.

</details>

## 5. Silent failure — why one paths file is a rule

Why so much ceremony for a few strings? Because of *how* a wrong path fails. A typo'd
filesystem path gives you `FileNotFoundError`. A typo'd YANG path in a Get gives you…
**an empty answer**. The router doesn't know you meant something else; it faithfully
reports that nothing lives at the address you asked for. Your code keeps running,
operating on nothing.

You already know this failure shape from Python — it's `dict.get` versus `dict[...]`.
Run the toy: same wrong key, one style crashes loudly, the other shrugs. gNMI Gets are
the shrugging kind.

In [ ]:
config_tree = {"qos": {"policer-templates": ["a2a-7"]}}

try:
    config_tree["qso"]                       # typo, loud style
except KeyError as err:
    print("✗ dict[...]  raised KeyError:", err, " ← you notice immediately")

print("→ dict.get :", config_tree.get("qso"), "        ← None. no error. you operate on nothing")

assert config_tree.get("qso") is None
print("\n✓ a gNMI Get on a typo'd path fails like .get — silently. one file to audit = the cure")

So the rule: **every path lives in `paths.py`, nowhere else.** When SR Linux upgrades and
renames a node, there is exactly one file to diff against the new schema — not a dozen
copies drifting inside function bodies.

One more one-liner before we meet the hands: **`json_ietf`**. When gNMI carries YANG data
as JSON, it follows RFC 7951 — the official rules for that encoding — and gNMI calls that
flavor `json_ietf`. SR Linux *rejects* the default encoding outright (docs/07, gotcha #2),
so every single call in netctl passes `encoding="json_ietf"`. Count them:

In [ ]:
prov_src = (ROOT / "netctl/src/netctl/provisioner.py").read_text()
hits = [ln.strip() for ln in prov_src.splitlines() if "json_ietf" in ln]
for ln in hits:
    print("  ", ln)

assert len(hits) >= 5
print(f"\n✓ {len(hits)} calls, {len(hits)} times json_ietf — no exceptions, ever")

## 6. `MockProvisioner` — hands that only remember

Here's the problem the mock solves: CI has no router. Your laptop (probably) has no
router. But the controller, the skeleton, and half the test suite need *something* on
the other end of the `NetworkProvisioner` port. So netctl ships two implementations of
the same port, side by side: `GnmiProvisioner` (configures real routers, §12) and
`MockProvisioner` — a **recording double** that satisfies the identical interface but
just writes down what it was asked to do.

You've already driven it, unknowingly: 05's `FakeNet` is a plain alias for this exact
class (05's swap map showed you the `assert FakeNet is MockProvisioner`). Here is the
real source — all of it. Notice what's *missing*: no gNMI, no network, no inheritance.

In [ ]:
from netctl.mock import MockProvisioner

print(inspect.getsource(MockProvisioner))

Two shelves and four methods:

- `self.applied: dict[str, dict]` — one entry per live session: *what a router would
  have been told*. (The `dict[str, dict]` annotations on instance attributes are the
  type hints from 01, applied to `self`.)
- `self.torn_down: list[str]` — every teardown call, appended forever. Tests read this
  shelf to prove teardown *happened*, even after `applied` is empty again.
- Every method returns `ApplyResult` — the tiny frozen pydantic model from 02
  (`ok: bool`, `detail: str`). Results are **values**, not exceptions; keep that in
  mind for §13 where the real hands make the same choice.

Drive it with the canonical numbers from 04 — `RESOLVED_PATH` is the concrete
device-and-interfaces bundle the controller produced from ticket #7's opaque
`resource_id` via the resource map (ADR-005, chapter 08). Rule 6: netctl never sees a
resource id or a ticket — only names it can act on.

In [ ]:
from a2a_interfaces import fixtures as fx

SESSION = "course-7"                       # our session for ticket #7's purchase

mock = MockProvisioner()
result = mock.apply_bandwidth(SESSION, fx.RESOLVED_PATH, fx.CAPACITY_50_MBPS, fx.QOS_CLASS)

print("returned      →", result)
print("mock.applied  →", mock.applied[SESSION])
print("mock.torn_down→", mock.torn_down)

assert result.ok
assert mock.applied[SESSION]["capacity_bps"] == 50_000_000     # Ada's 50 Mbps, in bps
assert mock.applied[SESSION]["path"].device == "srl1"
assert mock.torn_down == []
print("\n✓ recorded, not provisioned: device srl1, ingress ethernet-1/1, 50 Mbps")

**✏️ Your turn 6.1 — half the bandwidth, same hands (modify and observe)**

On a *fresh* mock (so §7's shelves stay untouched), record a session `course-7-echo`
that buys `half` — 25 Mbps — instead of the full 50. Change one argument, rerun, and
watch the shelf. Success: the recorded `capacity_bps` reads exactly `25,000,000`.

In [ ]:
mock2 = MockProvisioner()                     # fresh hands — §7 needs `mock` untouched
half = fx.CAPACITY_50_MBPS // 2

# TODO: make the echo session carry `half` instead of the full 50 Mbps
res2 = mock2.apply_bandwidth("course-7-echo", fx.RESOLVED_PATH, fx.CAPACITY_50_MBPS, fx.QOS_CLASS)
print("recorded capacity →", f"{mock2.applied['course-7-echo']['capacity_bps']:,} bps")

# un-comment once the shelf shows half:
# assert mock2.applied["course-7-echo"]["capacity_bps"] == 25_000_000

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
res2 = mock2.apply_bandwidth("course-7-echo", fx.RESOLVED_PATH, half, fx.QOS_CLASS)
assert mock2.applied["course-7-echo"]["capacity_bps"] == 25_000_000
```

The mock records whatever it's told, verbatim — which is exactly what makes it usable
as evidence in tests: the shelf *is* the call.

</details>

## 7. Teardown, twice — rule 8 in your own hands

Rule 8: **teardown is idempotent** — calling it twice is a success, not an error (05
introduced the word; here you own the mechanism). Why the system *needs* this: two
independent paths can end the same session — the clock expiring (`tick`) and Bell
revoking the ticket — and nothing guarantees only one fires. A controller that crashed
mid-teardown will also retry after restart. If the second call errored, every one of
those ordinary situations would become a fault.

Watch the shelves across three calls: a real teardown, the *same* teardown again, and a
teardown of a session that never existed.

In [ ]:
first  = mock.teardown(SESSION)
second = mock.teardown(SESSION)                    # again — rule 8 says: success
ghost  = mock.teardown("never-existed")            # never applied — still success

print("first   →", first)
print("second  →", second, "  ← not an error")
print("ghost   →", ghost,  "  ← not an error either")
print("applied   →", mock.applied)
print("torn_down →", mock.torn_down)

assert first.ok and second.ok and ghost.ok
assert mock.applied == {}                          # the record is gone…
assert mock.torn_down == [SESSION, SESSION, "never-existed"]   # …but every call is remembered
print("\n✓ rule 8, demonstrated: teardown converges on 'gone' and never complains")

The whole trick is one Python idiom: `dict.pop(key, default)`. With a default, `pop`
removes the key *if present* and never raises — remove-if-there, exactly idempotency's
shape. Without a default, a missing key is a `KeyError`. The mock's teardown uses
`self.applied.pop(session_id, None)`; the toy below shows both behaviors so you can see
what that one extra argument buys.

In [ ]:
shelf = {"course-7": "policer config"}

print("pop with default   →", shelf.pop("course-7", None))    # removes, returns the value
print("pop again, default →", shelf.pop("course-7", None), "  ← gone already: None, no drama")

shelf["course-7"] = "policer config"
shelf.pop("course-7")                                # no default this time…
try:
    shelf.pop("course-7")                            # …so the second call:
except KeyError as err:
    print("✗ pop, no default  → KeyError:", err, " ← the bug §11 plants on purpose")

assert shelf == {}
print("\n✓ one default argument is the entire distance between rule 8 and a crash")

**✏️ Your turn 7.1 — the fourth teardown (predict, then check)**

`mock`'s shelves right now: `applied` is empty, `torn_down` holds three entries.
Predict — before running — what a *fourth* `teardown(SESSION)` returns and how long
`torn_down` gets. Success: both predictions match and you can say which shelf
converges and which one only ever grows.

In [ ]:
prediction_ok  = "?"        # TODO: "True" or "False" — BEFORE running
prediction_len = 0          # TODO: len(mock.torn_down) after the call

fourth = mock.teardown(SESSION)
print("fourth call →", fourth)
print("torn_down   →", mock.torn_down)

# un-comment once your predictions match:
# assert fourth.ok and len(mock.torn_down) == 4

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
prediction_ok  = "True"
prediction_len = 4
assert fourth.ok and len(mock.torn_down) == 4
```

`applied` converges (pop-with-default finds nothing, removes nothing) while
`torn_down` appends unconditionally — idempotent *effect*, complete *history*.

</details>

## 8. Two hands, one badge

The structural-typing beat (03's whole lesson, one cell): both provisioners satisfy the
same `@runtime_checkable` Protocol — no inheritance, no registration, just the right
method shapes. `GnmiProvisioner({})` is safe to build anywhere: the empty dict means
"I know zero routers", and nothing dials out until a method needs a device.

In [ ]:
from netctl.provisioner import GnmiProvisioner

pair = [("MockProvisioner", MockProvisioner()), ("GnmiProvisioner", GnmiProvisioner({}))]
for name, prov in pair:
    ok = isinstance(prov, NetworkProvisioner)
    print(f"{name:16} satisfies NetworkProvisioner → {ok}")
    assert ok

print("\n✓ same hole, two plugs — the controller from 08 cannot tell which one it holds")

## 9. Rule 7 — the police: one suite, both hands

03 ended on a warning (§17–18 there): `isinstance` against a Protocol checks **names
only**. A mock whose `teardown` exists but *behaves differently* — raises on the second
call, say — wears the badge while poisoning every test built on it. The controller would
pass CI against the mock and fail at 2 a.m. against the router.

Rule 7 is the counter-measure: **mocks implement the same Protocol as real adapters and
pass the same shared contract-test suite.** Not a similar suite. The same test
functions, run twice.

The machinery lives in `netctl/tests/`, and it's two small files. First the fixture file,
`conftest.py` — pytest's name for "shared test ingredients live here":

In [ ]:
print((ROOT / "netctl/tests/conftest.py").read_text())

Glossing the pytest machinery (you've *run* pytest before; here's the 60-second tour of
how it works): pytest collects functions named `test_*` and runs each one. A **fixture**
is a named ingredient — a test that declares a parameter called `provisioner` gets
whatever the `provisioner` fixture yields. And `params=["mock", "gnmi"]` is the trick
that *is* rule 7: pytest runs **every test once per param** — once handed a
`MockProvisioner`, once handed a `GnmiProvisioner` aimed at the live lab.

Details worth the read:

- the gnmi leg **skips, never fails**, without a lab (`pytest.skip(...)`) — CI runs the
  mock leg green, your machine with the lab up runs both;
- `scope="session"` builds ONE `GnmiProvisioner` for the entire run — SR Linux
  rate-limits gNMI *connections* (60/min), so a dial-per-test suite would lock itself
  out of its own router;
- the gnmi leg tears down after every test *even on failure* — and that cleanup is safe
  to run unconditionally precisely because of rule 8.

Now the suite itself. The cell loads it with `importlib` — the by-hand version of
`import`, needed because `tests/` is deliberately not an importable package — then shows
the two tests that matter most here.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "contract_suite", ROOT / "netctl/tests/test_provisioner_contract.py"
)
suite = importlib.util.module_from_spec(spec)
spec.loader.exec_module(suite)

tests = [n for n in dir(suite) if n.startswith("test_")]
print("the contract, as test names:")
for n in tests:
    print("  ", n)

print()
print(inspect.getsource(suite.test_satisfies_the_port))
print(inspect.getsource(suite.test_teardown_is_idempotent))
assert "test_teardown_is_idempotent" in tests

**✏️ Your turn 9.1 — run two police checks by hand (write a tiny thing)**

A contract test is just a function — no pytest required. Call `suite.test_health` and
`suite.test_teardown_of_unknown_session_succeeds` on `fresh` yourself. Success: both
return silently (that's what "passed" means), and `torn_down` shows exactly
`['never-existed']`.

In [ ]:
fresh = MockProvisioner()

# TODO: run two police checks by hand — each is just a function taking `provisioner`:
#   suite.test_health(fresh)
#   suite.test_teardown_of_unknown_session_succeeds(fresh)
print("shelves → applied:", fresh.applied, "· torn_down:", fresh.torn_down)

# un-comment once both ran silently:
# assert fresh.torn_down == ["never-existed"]

<details><summary>✅ Solution 9.1 — peek only after trying</summary>

```python
suite.test_health(fresh)
suite.test_teardown_of_unknown_session_succeeds(fresh)
assert fresh.torn_down == ["never-existed"]
```

pytest adds collection and the mock/gnmi parametrization, but the *contract* is plain
functions asserting on the port's surface — which is why §11 can aim one at a liar
directly.

</details>

## 10. Run the police, for real

Let's execute the suite exactly as CI does. The cell shells out to pytest with
`subprocess.run` — Python's "run another program and capture what it prints" (the same
tool `lab_ipv4` uses in §18 to ask docker questions).

What to expect in the output: **7 passed, 6 skipped**. The 6 parametrized tests run on
the mock leg (6 passed) plus one gNMI-only test of the loud-error path (§16) = 7; the
same 6 on the gnmi leg skip because no lab is running. With the lab up, the skips
become passes — same command, thirteen green.

In [ ]:
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "-m", "pytest", str(ROOT / "netctl/tests/test_provisioner_contract.py"), "-q"],
    capture_output=True, text=True, cwd=ROOT,
)
print(proc.stdout.strip())

assert proc.returncode == 0, proc.stdout + proc.stderr
assert "passed" in proc.stdout
print("\n✓ the mock leg is green — and the gnmi column of the same table is waiting for a lab")

## 11. Break a mock on purpose

Time to watch the police catch someone. `ForgetfulMock` below is `MockProvisioner` with
one character-sized bug: its teardown pops **without** the default — §7's loud style. On
the first teardown it works perfectly; only a *second* teardown of the same session
explodes. Precisely the divergence rule 7 exists for.

Predict before you run: (a) does `isinstance` still say it's a `NetworkProvisioner`?
(b) which contract test goes red, and with what exception?

In [ ]:
from a2a_interfaces import ApplyResult

class ForgetfulMock(MockProvisioner):
    def teardown(self, session_id: str) -> ApplyResult:
        self.applied.pop(session_id)          # ← no default: second call will KeyError
        self.torn_down.append(session_id)
        return ApplyResult(ok=True)

bad = ForgetfulMock()
print("still wears the badge →", isinstance(bad, NetworkProvisioner), " ← shape is fine!")
assert isinstance(bad, NetworkProvisioner)

suite.test_teardown_is_idempotent(MockProvisioner())      # the honest mock: passes silently
print("honest mock           → test_teardown_is_idempotent passed silently")

try:
    suite.test_teardown_is_idempotent(bad)                # the same test function, the liar
except KeyError as err:
    print("forgetful mock        → ✗ KeyError:", err, " ← caught by the SAME test")

print("\n✓ isinstance checks names; the contract suite checks behavior. you need both")

**✏️ Your turn 11.1 — a different liar (break and read the error)**

`BoolMock` below lies differently: `apply_bandwidth` does the work but returns a bare
`True` instead of an `ApplyResult`. Aim
`suite.test_apply_bandwidth_then_teardown_roundtrip(liar)` at it inside the `try`.
Success: the error *name* tells you the suite probed `.ok` on a bool — behavior the
badge never checks.

In [ ]:
class BoolMock(MockProvisioner):
    def apply_bandwidth(self, session_id, path, capacity_bps, qos_class):
        super().apply_bandwidth(session_id, path, capacity_bps, qos_class)
        return True                            # ← the lie: a bare bool, not an ApplyResult

liar = BoolMock()
print("still wears the badge →", isinstance(liar, NetworkProvisioner))

# TODO: replace `pass` with: suite.test_apply_bandwidth_then_teardown_roundtrip(liar)
try:
    pass
except Exception as err:
    print(f"✗ caught {type(err).__name__}: {err}")

# un-comment once you've read the error:
# caught = None
# try:
#     suite.test_apply_bandwidth_then_teardown_roundtrip(BoolMock())
# except Exception as e:
#     caught = type(e).__name__
# assert caught == "AttributeError"           # `.ok` probed on a bool — behavior, not names

<details><summary>✅ Solution 11.1 — peek only after trying</summary>

```python
try:
    suite.test_apply_bandwidth_then_teardown_roundtrip(liar)
except Exception as err:
    print(f"✗ caught {type(err).__name__}: {err}")   # AttributeError: 'bool' ... 'ok'
```

The test's first move is `assert result.ok, result.detail` — a bool has no `.ok`, so
the suite catches in one line a divergence `isinstance` structurally cannot see.

</details>

One sentence to keep: **the Protocol is the badge, the contract suite is the police** —
a mock that diverges at the port isn't a convenience, it's a bug, and
`test_teardown_is_idempotent[mock]` is where that bug turns red. Everything the mock
promised in §6–8 is only trustworthy because §10's run holds.

## 12. The real hands, part 1: connecting honestly (`connect.py`)

Now the adapter that earns the name. Before `GnmiProvisioner` can Set anything it must
*reach* the router, and this lab makes that genuinely awkward — awkward in ways worth
learning, because they're the texture of real infrastructure work.

Gloss first: **TLS** is the encryption-plus-identity layer under `https://`. The
identity half works with **certificates** — a signed statement that "this public key
belongs to this name" — normally vouched for by an authority your system already
trusts. The lab router has no authority: it generates a **self-signed** certificate
(it vouches for itself), and Python's grpc refuses to blindly accept those — pygnmi's
`skipverify` flag *still verifies underneath* and fails (docs/07 appendix; a documented
dead end, so you don't re-walk it).

`connect.py` is the one module allowed to know the workaround. Read it whole:

In [ ]:
import netctl.connect as connect_mod

print(inspect.getsource(connect_mod))

The recipe is called **trust-on-first-use** (TOFU): fetch the router's certificate over
plain TLS (`ssl.get_server_certificate` — note it takes a single *tuple* argument), save
it to a temp file, and hand it to grpc as the trusted root — a self-signed cert is its
own authority. Then the subtlest line in the file:

```python
override=target.tls_name or None,
```

The cert names its owner `srl1` — the node's short name, **not** the container hostname.
`override` makes grpc verify against that name. Set `tls_name` wrong (or leave it empty:
the `or None` idiom turns `""` into `None`) and verification fails — but it *surfaces as
a timeout*, not a certificate error. That trap cost this repo real hours (M3.1); the
docstring on `GnmiTarget` is its tombstone.

`GnmiTarget` itself is a frozen dataclass (01): the coordinates of one router, immutable
so nothing can quietly rewrite a port or password after construction. Poke it:

In [ ]:
import dataclasses

from netctl.connect import GnmiTarget

t = GnmiTarget(host="203.0.113.9", tls_name="srl1")
print(t)                                             # repr for free: every field visible
assert t.port == 57400                               # gNMI's customary port (network kind)
assert t.password == "NokiaSrl1!"                    # SR Linux default — lab only, obviously

try:
    t.port = 9999
except dataclasses.FrozenInstanceError as err:
    print("✗ mutation blocked:", err)

print("\n✓ coordinates are a value, not a mutable bag — frozen=True is the lock")

**✏️ Your turn 12.1 — what does grpc actually receive? (predict, then check)**

Build a `GnmiTarget` *without* naming `tls_name` and predict two reprs before running:
the field's default value, and what the `target.tls_name or None` line from
`connect()` hands to grpc. Success: both predictions match and you can say why the
`or None` is there at all.

In [ ]:
bare = GnmiTarget(host="203.0.113.9")          # tls_name left at its default

prediction_default = "?"    # TODO: bare.tls_name — None? "" ? "srl1"?
prediction_grpc    = "?"    # TODO: bare.tls_name or None — what reaches grpc?

print("default tls_name →", repr(bare.tls_name))
print("grpc override    →", repr(bare.tls_name or None))
print("srl1's override  →", repr(t.tls_name or None))

# un-comment once your predictions match:
# assert bare.tls_name == "" and (bare.tls_name or None) is None

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
prediction_default = "''"      # the dataclass default is an empty string
prediction_grpc    = "None"    # `"" or None` → None: no override requested
assert bare.tls_name == "" and (bare.tls_name or None) is None
```

grpc's `override` treats `None` as "verify against the hostname" and `""` as a
literal (never-matching) name — the idiom converts "not set" into the value grpc
means by it.

</details>

## 13. The real hands, part 2: `apply_bandwidth`

Here it is — §2's hand-typed recipe, reborn as one method. Read it top to bottom and
keep the sr_cli lines in your head; you'll recognize every leaf name.

In [ ]:
print(inspect.getsource(GnmiProvisioner.apply_bandwidth))

Walking it, decision by decision:

- **`name = _template_name(session_id)`** → `a2a-course-7`. The template is named after
  the session — remember this; it's the entire teardown design (§14).
- **`rate_kbps = max(capacity_bps // 1000, 1)`** — the gNMI leaf wants kbps, the port
  speaks bps (04's bits-vs-bytes discipline). `//` floors, and `max(..., 1)` keeps a
  sub-1000-bps request from flooring to zero — a router would reject rate 0.
- **the `template` dict** is the policer: peak/committed rate, and a burst allowance
  (`_BURST_BYTES = 125_000` — 20 ms of traffic at 50 Mbps, enough for TCP to breathe).
  Nested dict/list literals *are* the JSON payload; no serialization step to see.
- **`{"drop": [None]}`** is the strangest line in the file. In YANG, `drop` is an
  *empty leaf* — a flag that's either present or absent, with no value. RFC 7951 says:
  encode presence as `[null]`. Python's `None` becomes JSON's `null`, so `[None]` it is.
  Write `{}` instead and the router rejects the Set.
- **one `.set(update=[(path1, payload1), (path2, payload2)])`** — template and
  attachment land in a single gNMI transaction: both or neither, never a dangling half.
- **`except Exception: return ApplyResult(ok=False, detail=...)`** — errors as values,
  same as the mock. The port *reports*; the caller (the controller) decides. One shape
  of honesty at the boundary.

Pin the checkable claims:

In [ ]:
import json

from netctl.provisioner import _template_name       # _private: peeking to learn

src = inspect.getsource(GnmiProvisioner.apply_bandwidth)
assert '"violate-action": {"drop": [None]}' in src   # the RFC 7951 empty leaf, verbatim

print("Python {'drop': [None]}  → JSON", json.dumps({"drop": [None]}), " ← [null], not {}")

rate = max(fx.CAPACITY_50_MBPS // 1000, 1)
print(f"{fx.CAPACITY_50_MBPS:,} bps // 1000       → {rate:,} kbps   ← the leaf's unit")
assert rate == 50_000
assert max(999 // 1000, 1) == 1                      # the floor's guard rail

print("_template_name('course-7') →", repr(_template_name("course-7")))
assert _template_name("course-7") == "a2a-course-7"
print("\n✓ recipe → payload verified without a router in sight")

**✏️ Your turn 13.1 — make the floor shave something (modify and observe)**

At 50,000,000 bps the `// 1000` floor loses nothing. Change `paid_bps` to
`1_234_567`, rerun, and watch what the kbps leaf silently drops. Success: the
un-commented assert passes, and you can say which *direction* the floor errs —
does Ada ever get more than she paid for?

In [ ]:
paid_bps = fx.CAPACITY_50_MBPS      # TODO: change to 1_234_567 and rerun
leaf_kbps = max(paid_bps // 1000, 1)
lost_bps  = paid_bps - leaf_kbps * 1000
print(f"paid {paid_bps:,} bps → leaf {leaf_kbps:,} kbps → {lost_bps} bps shaved by the floor")

# un-comment once the floor visibly shaves something:
# assert (leaf_kbps, lost_bps) == (1234, 567)

<details><summary>✅ Solution 13.1 — peek only after trying</summary>

```python
paid_bps = 1_234_567
assert (leaf_kbps, lost_bps) == (1234, 567)
```

`//` always rounds *down*, so the enforced rate is never above the purchased one —
the provider under-delivers by at most 999 bps rather than ever over-granting.

</details>

## 14. Stateless teardown — the router remembers, the process doesn't

Now the design decision that makes this adapter genuinely elegant. `GnmiProvisioner`
keeps **no record** of what it has applied — no `self.applied` dict like the mock. So
how can `teardown("course-7")` know what to remove?

It *asks the router*. Every piece of config this adapter writes is named
`a2a-<session_id>`, so teardown Gets the config tree, finds everything carrying its
session's name, and deletes it. The state lives on the device — which means teardown
works from a **freshly restarted process**, and a second call simply finds nothing and
succeeds. Rule 8 isn't bolted on here; it falls out of the design.

Read `teardown` and the finder it calls:

In [ ]:
print(inspect.getsource(GnmiProvisioner.teardown))
print(inspect.getsource(GnmiProvisioner._session_config_on))

Two details in `_session_config_on` deserve slow reading:

- **Attachment-first ordering.** The delete list collects interface *attachments* before
  the *template*: the router refuses to delete a template something still references,
  even inside one transaction. Reverse the order and teardown fails with the config
  stranded on the device.
- **Defensive navigation.** gNMI responses are deep dicts where whole branches may be
  missing or `None`. Two idioms keep the walk crash-free: chained
  `.get("x", {}).get("y", {})` (a missing branch yields `{}`, and the walk continues),
  and `... or []` (a present-but-`None` branch becomes an empty list, and the `for` loop
  runs zero times). Run the toy to feel both:

In [ ]:
response = {"notification": [{"update": None}]}          # present, but None — gNMI does this

for update in response["notification"][0].get("update") or []:
    print("never reached")
print("→ `or []` : looped zero times over a None branch — no crash")

interface = {"interface-id": "ethernet-1/1.0"}           # attachment with no `input` branch
attached = interface.get("input", {}).get("policer-templates", {}).get("policer-template")
print("→ chained .get with {} defaults :", attached, "  ← None means 'not ours', not 'boom'")

assert attached is None
print("\n✓ every missing branch degrades to 'nothing found' — teardown stays calm")

## 15. `_denamespace` — the prefix stripper (and your first recursion)

One more trap between a Get response and those calm lookups. RFC 7951 requires that a
key be **prefixed with its YANG module name** whenever the node was grafted in from
another module (YANG calls that an *augment*). So the router doesn't answer with
`"policer-template"` — it answers with `"srl_nokia-qos:policer-template"`. A plain
dict lookup on the unprefixed name **silently misses** (§5's failure shape again!), and
a teardown built on such lookups would report "nothing to remove" while the policer
sits on the router forever. Silent leftover config — the worst kind.

The fix walks the whole response and strips prefixes from every key at every depth.
"Every depth" calls for **recursion** — a function that calls *itself* on each nested
piece until it hits something that isn't nested. If you haven't met it: toy first.

In [ ]:
def count_leaves(node):
    """How many non-container values hide inside arbitrarily nested dicts/lists?"""
    if isinstance(node, dict):
        return sum(count_leaves(value) for value in node.values())   # ← calls itself
    if isinstance(node, list):
        return sum(count_leaves(item) for item in node)              # ← calls itself
    return 1                                                         # ← the bottom: a leaf

nested = {"a": [1, 2, {"b": 3}], "c": {"d": [4, 5]}}
print("leaves in", nested, "→", count_leaves(nested))
assert count_leaves(nested) == 5
print("✓ dispatch on type, recurse into containers, stop at leaves — that exact shape is next")

**✏️ Your turn 15.1 — recursion with no bottom (break and read the error)**

`snake` below contains *itself*, so `count_leaves` can never reach a leaf. Call
`count_leaves(snake)` inside the `try` and read what Python does about it. Success:
the error name tells you the recursion ran out of stack, not out of data — and you
can name which branch of `count_leaves` would have stopped it.

In [ ]:
snake = {}
snake["tail"] = snake        # a dict that contains itself — there is NO bottom

print("a healthy call first →", count_leaves({"a": [1, 2], "b": 3}), "leaves")

# TODO: replace `pass` with: count_leaves(snake)
try:
    pass
except RecursionError:
    print("✗ RecursionError — the recursion never found a leaf to stop at")

# un-comment once you've seen it fire:
# caught = None
# try:
#     count_leaves(snake)
# except RecursionError:
#     caught = "RecursionError"
# assert caught == "RecursionError"

<details><summary>✅ Solution 15.1 — peek only after trying</summary>

```python
try:
    count_leaves(snake)
except RecursionError:
    print("✗ RecursionError — the recursion never found a leaf to stop at")
```

Every recursive function lives or dies by a reachable base case — the `return 1` leaf
branch. `snake` makes that branch unreachable, so Python's depth limit is what
finally stops the spiral. (Real gNMI responses are trees, never cycles — the danger
here is the *idea*, not the router.)

</details>

Now the real one — same three-branch skeleton, but instead of counting it *rebuilds* the
structure with cleaned keys. `key.split(":", 1)[-1]` splits on the first colon and keeps
the last piece: `"srl_nokia-qos:policer-template"` → `"policer-template"`, and a key
with no colon survives untouched. Watch the silent miss happen on the raw response, then
disappear after stripping:

In [ ]:
from netctl.provisioner import _denamespace          # _private: peeking to learn

print(inspect.getsource(_denamespace))

raw = {"srl_nokia-qos:policer-template": [{"name": "a2a-7", "srl_nokia-qos:policer": []}]}
print("raw lookup   →", raw.get("policer-template"), "      ← the silent miss")

clean = _denamespace(raw)
print("cleaned      →", clean)
print("clean lookup →", clean["policer-template"][0]["name"])

assert raw.get("policer-template") is None
assert clean["policer-template"][0]["name"] == "a2a-7"
assert "policer" in clean["policer-template"][0]     # nested key stripped too — recursion at work
print("\n✓ strip once, look up plainly everywhere after")

## 16. Rule 8 with zero routers

Statelessness has a delightful corollary you can run *right now, with no lab*: a
`GnmiProvisioner` that knows zero devices loops over nothing in teardown and correctly
reports "nothing to remove" — twice, identically. The real class, really executing its
teardown logic, headless.

And the flip side — asked to *apply* onto a device it was never told about, it answers
loudly. Inside `_client` sits `raise KeyError(...) from None` (the `from None` mutes
Python's "during handling of the above exception" chaining noise), and `apply_bandwidth`
converts that to an `ok=False` result whose detail names the devices it *does* know.
The contract suite pins this too (`test_unknown_device_is_a_loud_error`).

In [ ]:
from a2a_interfaces import ResolvedPath

empty = GnmiProvisioner({})                              # knows no routers at all
d1 = empty.teardown("course-7")
d2 = empty.teardown("course-7")
print("teardown #1 →", d1)
print("teardown #2 →", d2, "  ← rule 8, real class, zero infrastructure")
assert d1.ok and d2.ok and d1.detail == "nothing to remove"

lonely = GnmiProvisioner({"srl1": GnmiTarget(host="127.0.0.1")})
res = lonely.apply_bandwidth(
    "course-7", ResolvedPath(device="srl9", ingress_if="e1", egress_if="e2"),
    fx.CAPACITY_50_MBPS, fx.QOS_CLASS,
)
print("unknown dev →", res.detail)
assert not res.ok and "srl9" in res.detail and "srl1" in res.detail
print("\n✓ quiet where quiet is right (rule 8), loud where guessing would be dangerous")

## 17. ADR-006 — the missing ASIC (the honesty section)

Time for the most important candor in this thesis. In a real router, forwarding and rate
limiting happen in an **ASIC** — a dedicated chip that processes packets at line rate.
The *control plane* (the software that accepts config) programs the ASIC (the
*datapath*), and "committed" means "enforced".

Containerized SR Linux has no ASIC. M2.2 measured what its software datapath actually
does with a committed policer, and the answer is brutal: the config is **accepted,
stored, readable back — and not enforced**. A 100 Mbit/s stream crossed a 50 Mbit/s
policer with 0% loss; the policer's own state tree reported `peak-rate-kbps 0`.

ADR-006 is the decision about where the *missing half* of the policer lives. Read its
Decision section straight from the repo:

In [ ]:
adr = (ROOT / "docs/adr/006-lab-datapath-enforcement.md").read_text()
print(adr[adr.index("## Decision"): adr.index("## Alternatives")].strip())

assert "not enforce it" in adr                    # the measured, admitted gap
assert "tc tbf" in adr                            # the shim that fills it

Unpacking the decision:

- The gNMI-committed policer config **stays the single source of truth**. netctl and the
  controller speak only gNMI to the router; neither ever learns the shim exists — the
  last cell of this section proves that mechanically.
- A lab script, `netlab/mirror-policer-to-tc.sh`, reads the router's committed config
  and mirrors it into **`tc tbf`** — Linux's built-in traffic shaper (`tc` = traffic
  control, `tbf` = token bucket filter) — *inside srl1's own container network
  namespace*. Enforcement still physically happens at the router, where the ASIC would
  do it. The shim is the simulator's physics engine, not an architectural component.
- On real hardware you simply don't deploy the shim. Nothing else changes.

So what does "enforced" honestly mean in this lab? **Config on the router + one shim
tick = real, measured packet loss.** The numbers below are quoted from the M2.2
evidence file — real pasted iperf3 output, not paraphrase. `iperf3` is the standard
throughput-measuring tool; hostA blasts traffic at hostB through srl1.

In [ ]:
ev = (ROOT / "docs/evidence/M2.2.md").read_text()
print("every iperf3 result line in docs/evidence/M2.2.md:\n")
for line in ev.splitlines():
    if "Mbits/sec" in line:
        print("  ", line.strip())

assert "75.6 Mbits/sec" in ev     # before: the unshaped (CPU-bound) TCP ceiling
assert "99.5 Mbits/sec" in ev     # before: 100M UDP offer sails through, 0% loss
assert "47.7 Mbits/sec" in ev     # after policer + shim tick: THE PLATEAU
assert "(51%)" in ev              # after: half the 100M UDP offer dropped at srl1
print("\n✓ before ≈ 75–99 Mbit/s free-for-all → after ≈ 48 Mbit/s plateau (51% dropped) → teardown → ceiling back")

Why is this candor a *strength* rather than an embarrassment? Because the claim the
thesis actually needs — *an on-chain entitlement drives real device configuration, and
revoking it takes the service away* — is fully real: real gNMI, real router OS, real
config, measured throughput obeying it. The one simulated piece (the ASIC) is named,
measured, documented in an ADR, and implemented so that **no production code knows it
exists**. Prove that last claim mechanically — scan every line of netctl for any
knowledge of the shim:

In [ ]:
for py in sorted((ROOT / "netctl/src/netctl").glob("*.py")):
    text = py.read_text()
    for word in ("shim", "tbf", "mirror-policer"):
        assert word not in text, f"{py.name} mentions {word!r}!"
    print(f"✓ {py.name:15} — no 'shim', no 'tbf', no 'mirror-policer'")

print("\n✓ rule 6 intact: the hands speak gNMI to the router and nothing else")

**✏️ Your turn 17.1 — audit the bouncer too (write a tiny thing)**

Rule 6 says netctl never heard of the shim — but the *controller* mustn't have
either. Repoint the scan at `controller/src/controller` and rerun. Success: every
file passes the same three-word audit, and the un-commented assert proves you really
scanned the bouncer (its `domain.py` shows up).

In [ ]:
scan_dir = ROOT / "netctl/src/netctl"        # TODO: repoint at controller/src/controller
scanned = sorted(scan_dir.glob("*.py"))
for py in scanned:
    text = py.read_text()
    for word in ("shim", "tbf", "mirror-policer"):
        assert word not in text, f"{py.name} mentions {word!r}!"
print(f"✓ scanned {len(scanned)} files under {scan_dir.name} — the shim is unheard of")

# un-comment once the scan points at the controller:
# assert any(p.name == "domain.py" for p in scanned)   # proof you scanned the bouncer

<details><summary>✅ Solution 17.1 — peek only after trying</summary>

```python
scan_dir = ROOT / "controller/src/controller"
```

The shim's entire audience is `netlab/` shell scripts — no production package
(hands *or* bouncer) carries a single mention, which is what "delete the shim on
real hardware and nothing else changes" means mechanically.

</details>

## 18. Knock on the router — the guarded live section

Everything so far ran anywhere. This section *detects* the lab and goes live only if
srl1 is actually running — otherwise every cell prints a skip line and the full terminal
recipe below gets you there later.

Detection is `netctl.testing.lab_ipv4()`: it asks docker (via `subprocess.run`, §10's
tool) for the srl1 container's IPv4 address. Two deliberate design choices to notice in
its source: it returns **`None` instead of raising** when the lab is down — "lab down"
is an *answer* here, not an error — and it asks for the IPv4 specifically because the
lab's hostname entries are IPv6-only and Python's grpc won't dial those (docs/07
appendix). One more lab fact before dialing: SR Linux **rate-limits gNMI connections
(60/min)** — so we build ONE provisioner and reuse its cached connection, exactly like
the conftest in §9 did.

In [ ]:
import netctl.testing as nt

print(inspect.getsource(nt.lab_ipv4))

try:
    LAB_IP = nt.lab_ipv4()
except FileNotFoundError:                      # no docker CLI at all on this machine
    LAB_IP = None

if LAB_IP is None:
    print("→ lab status: DOWN — live cells below will skip (recipe follows)")
else:
    print(f"→ lab status: UP — srl1 answers at {LAB_IP} ({nt.LAB_NODE})")

**Live poke 1 — health and one real Get.** If the lab is up: build the provisioner
(TOFU connect from §12 happens on first use), ping it with `health()` (gNMI
Capabilities), then Get the `oper-state` leaf of `ethernet-1/1` — "is this interface
actually up?" — using the exact path builder from §4. Note the deep indexing into the
notification envelope; you met that shape in §14.

In [ ]:
if LAB_IP is None:
    print("skipped: needs the live lab — run:  containerlab deploy -t netlab/topology.clab.yml")
else:
    live = GnmiProvisioner({"srl1": GnmiTarget(host=LAB_IP, tls_name="srl1")})
    print("health() →", live.health())

    client = live._client("srl1")             # _private: reuse the ONE cached connection
    got = client.get(path=[paths.interface_oper_state("ethernet-1/1")], encoding="json_ietf")
    state = got["notification"][0]["update"][0]["val"]
    print("ethernet-1/1 oper-state →", state, "  ← a real answer from a real router OS")

**Live poke 2 — the full product cycle.** Apply Ada's 50 Mbps, then *read our own config
back off the device* with the same finder teardown uses — the `a2a-course-7` name found
on the router, not remembered in this process. Then teardown twice (rule 8, live) and
`close()` the cached connections. If you want the *physics* (the iperf plateau), that
needs the ADR-006 shim tick — it's in the recipe below and in
[`netctl_explore.ipynb`](../netctl_explore.ipynb), which asserts the plateau numbers.

In [ ]:
if LAB_IP is None:
    print("skipped: needs the live lab — the full recipe follows below")
else:
    res = live.apply_bandwidth("course-7", fx.RESOLVED_PATH, fx.CAPACITY_50_MBPS, fx.QOS_CLASS)
    print("apply    →", res.ok, "·", res.detail)

    found = live._session_config_on(live._client("srl1"), "a2a-course-7")
    print("on srl1  →", found, "  ← our work, FOUND on the device by name")

    print("teardown →", live.teardown("course-7").detail)
    print("again    →", live.teardown("course-7").detail, "  ← rule 8, against real iron")
    live.close()

### The terminal recipe (when you're ready for the physics)

Everything below runs in a terminal at the repo root. Budget ~1 minute for srl1 to boot.

```sh
# 1. boot the flight simulator (hostA ── srl1 ── hostB)
containerlab deploy -t netlab/topology.clab.yml

# 2. rule 7, both legs: the gnmi column stops skipping (§10's table, all green)
uv run pytest netctl/tests/test_provisioner_contract.py -q

# 3. one raw Get with gnmic (a CLI gNMI client) — json_ietf or SR Linux hangs up (§5)
gnmic -a clab-a2a-srl1:57400 -u admin -p 'NokiaSrl1!' --skip-verify -e json_ietf \
  get --path "/interface[name=ethernet-1/1]/oper-state"

# 4. telemetry, raw (§19's product, seen with your own eyes): the router narrates
#    its own counters every 10 s — start an iperf3 in another terminal and watch
#    in-octets climb by ~70 MB per sample
gnmic -a clab-a2a-srl1:57400 -u admin -p 'NokiaSrl1!' --skip-verify -e json_ietf \
  subscribe --path "/interface[name=ethernet-1/1]/statistics" \
  --mode stream --stream-mode sample --sample-interval 10s

# 5. the M2.2 physics, end to end:
docker exec clab-a2a-hostA iperf3 -c 10.10.2.10 -u -b 100M   # BEFORE: ~99 Mbit/s, 0% loss

uv run python -c "
from a2a_interfaces.fixtures import CAPACITY_50_MBPS, QOS_CLASS, RESOLVED_PATH
from netctl.connect import GnmiTarget
from netctl.provisioner import GnmiProvisioner
from netctl.testing import lab_ipv4
prov = GnmiProvisioner({'srl1': GnmiTarget(host=lab_ipv4(), tls_name='srl1')})
print(prov.apply_bandwidth('terminal-demo', RESOLVED_PATH, CAPACITY_50_MBPS, QOS_CLASS))
"

./netlab/mirror-policer-to-tc.sh                             # ADR-006's missing ASIC (§17)
docker exec clab-a2a-hostA iperf3 -c 10.10.2.10 -u -b 100M   # AFTER: ~48 Mbit/s, ~51% dropped

uv run python -c "
from netctl.connect import GnmiTarget
from netctl.provisioner import GnmiProvisioner
from netctl.testing import lab_ipv4
prov = GnmiProvisioner({'srl1': GnmiTarget(host=lab_ipv4(), tls_name='srl1')})
print(prov.teardown('terminal-demo'))
"
./netlab/mirror-policer-to-tc.sh                             # shim follows the config down
docker exec clab-a2a-hostA iperf3 -c 10.10.2.10 -u -b 100M   # ceiling is back

# 6. then re-run this notebook (live cells light up) and netctl_explore.ipynb
containerlab destroy -t netlab/topology.clab.yml             # fold the lab away
```

A worthwhile detour while the lab is up: typo one segment of a path (say
`policer-template**s**` → `policer-template`) in a raw `gnmic get` — and watch it return
*nothing* instead of erroring. That silence is §5's whole argument, live.

## 19. The second product: telemetry — a config *right* on the device (ADR-007)

The marketplace sells two things. Bandwidth you now know inside out. The second product
— story chapter 7, canonical ticket **#8** — is **telemetry**: Ada pays to watch the
router's interface counters stream to her own **collector** (her receiving endpoint,
`10.0.0.50:57000` in the fixtures).

The design question ADR-007 answers: what does Bell actually *deliver*? Not a data feed
run by some provider-side forwarder process. The ticket buys **the right to a config
write on the device**: `apply_telemetry` writes a gNMI dial-out *destination* onto srl1
(SR Linux's `grpc-tunnel destination` — remember `paths.telemetry_destination` from §4),
pointing at Ada's collector. The **router itself** exports toward her.

That makes telemetry perfectly symmetric with bandwidth — same shape, second verse: the
token authorizes a config write · the config lives ON the device, readable back · it's
named `a2a-<session>` · teardown finds and removes it, statelessly (rule 8). Read it:

In [ ]:
print(inspect.getsource(GnmiProvisioner.apply_telemetry))

One string-surgery idiom in there is new: splitting `"10.0.0.50:57000"` into host and
port. `str.rpartition(":")` cuts at the **last** colon (safer than the first — think
IPv6 addresses full of colons) and always returns a 3-tuple `(before, separator,
after)`, which the code unpacks as `host, _, port` — the underscore is Python's
customary name for "I must receive this, but I won't use it". Then a ternary
(01) turns the port string into an int, with a default if it's missing or garbage:

In [ ]:
endpoint = fx.TELEMETRY_NEED.collector_endpoint          # the canonical collector (04)
host, _, port = endpoint.rpartition(":")
print(f"{endpoint!r}.rpartition(':') → host={host!r}  sep={_!r}  port={port!r}")

port_num = int(port) if port.isdigit() else 57400        # the method's exact fallback line
print("as an int (with fallback)    →", port_num)

assert (host, port_num) == ("10.0.0.50", 57000)
no_port = "collector.example".rpartition(":")
print("no colon at all              →", no_port, " ← host lands in slot [2]!")
assert no_port == ("", "", "collector.example")
print("\n✓ rpartition never raises — the fallback ternary absorbs every weird endpoint")

**✏️ Your turn 19.1 — an IPv6 collector (predict, then check)**

Ada's collector could live at `[2001:db8::1]:57000` — five colons. Predict the host
that `rpartition(":")` extracts *before* running, then compare with what a naive
first-colon `partition` would have produced. Success: your prediction matches and the
word "last" now feels load-bearing.

In [ ]:
v6 = "[2001:db8::1]:57000"                   # a collector on an IPv6 address

predicted_host = "?"                         # TODO: your prediction BEFORE running

host6, _, port6 = v6.rpartition(":")
print(f"rpartition → host={host6!r}  port={port6!r}")
print("partition (FIRST colon) → host =", repr(v6.partition(":")[0]), " ← garbage")

# un-comment once your prediction matches:
# assert (host6, port6) == ("[2001:db8::1]", "57000")

<details><summary>✅ Solution 19.1 — peek only after trying</summary>

```python
predicted_host = "[2001:db8::1]"
assert (host6, port6) == ("[2001:db8::1]", "57000")
```

The port always follows the *last* colon, so cutting there survives any number of
colons inside the host — `partition` would have cut at `[2001` and fed garbage to
the int fallback.

</details>

Now honor ticket #8 with the mock — same call the controller makes, canonical values
from 04 throughout (`RESOLVED_NODE` is just `device="srl1"`: telemetry needs a node, not
an ingress/egress pair — that's why the port has both `ResolvedPath` *and*
`ResolvedNode`). Then tear down twice, because by now your hands do that on reflex.

In [ ]:
tmock = MockProvisioner()
res = tmock.apply_telemetry(
    "course-8",
    fx.RESOLVED_NODE,
    fx.TELEMETRY_NEED.sensor_paths,
    fx.TELEMETRY_NEED.collector_endpoint,
    fx.TELEMETRY_NEED.sample_interval_s,
)
print("returned →", res)
print("recorded →", tmock.applied["course-8"])

rec = tmock.applied["course-8"]
assert res.ok
assert rec["collector_endpoint"] == "10.0.0.50:57000"
assert rec["sensor_paths"] == ["/interface[name=ethernet-1/1]/statistics"]
assert rec["sample_interval_s"] == 10

assert tmock.teardown("course-8").ok and tmock.teardown("course-8").ok    # reflexes
print("\n✓ ticket #8 honored and torn down — the second product, same discipline as the first")

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| read the hand recipe, then its Python twin | provisioning: config writes make promises physical; CLI → gNMI | the controller calls it on activation (08 → 11) |
| built four YANG paths from `paths.py` | paths fail *silently*, so they live in ONE file | one diff target when SR Linux upgrades |
| counted seven `json_ietf`s | RFC 7951 is the only encoding SR Linux accepts | every gNMI call in the repo |
| applied + tore down on `MockProvisioner`, twice | rule 8: teardown converges, never complains | tick & revocation both fire safely (12) |
| watched `test_teardown_is_idempotent` catch `ForgetfulMock` | rule 7: the Protocol is the badge, the contract suite is the police | every mock/real pair in the repo |
| walked TOFU connect + frozen `GnmiTarget` | lab TLS honesty: self-signed cert as its own root, SNI = cert's own name | `connect()` on first live use |
| pinned `{"drop": [None]}` and the kbps floor | RFC 7951 empty leaf; payloads are nested literals | the Set that lands on srl1 |
| ran `GnmiProvisioner({}).teardown` headless | stateless teardown: config is FOUND by `a2a-<sid>` name | restart-safe cleanup, revocation (12) |
| stripped prefixes with `_denamespace` | YANG augments prefix keys; plain lookups silently miss | every Get response netctl parses |
| quoted 99.5 → 48.5 (51% dropped) from evidence | ADR-006: container accepts, doesn't enforce; the shim is the missing ASIC — and netctl provably never heard of it | honest claims in the finale (12) & the evaluation chapters (13–14) |
| honored ticket #8 with `apply_telemetry` | ADR-007: telemetry = a config RIGHT on the device, symmetric with bandwidth | the second product in the finale (12) |

## Experiments to try

Predict the outcome *before* running each:

- Run `ForgetfulMock` through the **whole** suite, not just one test: temporarily point
  §9's `spec_from_file_location` trick at a copy of `conftest.py` — or simpler, call each
  of the seven `suite.test_*` functions on a fresh `ForgetfulMock()` and count which ones
  survive. (Careful: some tests need the `provisioner` argument, one needs none.)
- `mock.apply_bandwidth(SESSION, ...)` twice with different capacities, no teardown
  between. What does `applied[SESSION]` hold, and is that the right port behavior — or
  something the contract suite *should* pin? (08's `E_CONFLICT` check is related.)
- Feed `_denamespace` a list of strings like `["a:b", "c"]`. What comes back, and which
  branch of the recursion handled each element?
- Deliberate breakage, lab edition: with the lab up, edit your *local copy* of
  `apply_bandwidth` to send `{"drop": {}}` instead of `{"drop": [None]}` and watch the
  router reject the Set — delivered politely as `ApplyResult(ok=False, detail=...)`,
  never an exception.

## Check yourself

1. A colleague proposes inlining the path strings into `provisioner.py` "since they're
   only used there". What failure mode gets worse, and why is it uniquely nasty?
2. Rule 8 says teardown twice must succeed. Where does that behavior *live* in
   `MockProvisioner`, and where does it live in `GnmiProvisioner`? (Two very different
   answers — one is an argument to `pop`, one is a whole design.)
3. `isinstance(ForgetfulMock(), NetworkProvisioner)` is `True`. Why is that the correct
   behavior for a Protocol, and what — precisely — is the mechanism that catches the lie?
4. What does "the policer is enforced" honestly mean in this lab, which component fills
   the ASIC's role, and how do you *prove* netctl doesn't know about it?
5. Ada's telemetry ticket #8 gets revoked mid-stream. Walk what teardown does on the
   router, and explain why a provider-side forwarder process would have made that story
   messier (ADR-007).

**Where this goes next:** `10_agents_the_brains.ipynb` — the LLM judgment slots (rule
1's only two homes): how the consumer decides to buy and the provider decides to quote,
with everything you've built so far as their deterministic world.

**Deeper dive:** the live-lab twin [`netctl_explore.ipynb`](../netctl_explore.ipynb) ·
the lab's own story [`docs/07-netlab.md`](../../../docs/07-netlab.md) (§6 the policer by
hand, §7 telemetry raw) ·
[`docs/adr/006-lab-datapath-enforcement.md`](../../../docs/adr/006-lab-datapath-enforcement.md)
· the police themselves:
[`netctl/tests/test_provisioner_contract.py`](../../../netctl/tests/test_provisioner_contract.py).